In [1]:
import math
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
def get_largest_leap(df: pd.DataFrame):
    return df["end_beat"].diff().abs().max()

In [3]:
def get_average_note_length(df: pd.DataFrame):
    return df["end_beat"].mean()

In [4]:
def get_most_common_note(df: pd.DataFrame):
    return df["end_beat"].value_counts(normalize=True).iloc[0]

In [5]:
def get_average_beat_gap(df: pd.DataFrame):
    beat_gap = df["start_beat"].diff().dropna()
    return beat_gap.mean()

In [6]:
def get_unique_note_values(df: pd.DataFrame):
    return df["end_beat"].nunique()

In [7]:
def get_number_of_instruments(df: pd.DataFrame):
    return df["instrument"].nunique()

In [8]:
def get_pitch_range(df: pd.DataFrame):
    return df["end_beat"].max() - df["end_beat"].min()

In [9]:
def get_pitch_std(df: pd.DataFrame):
    return df["end_beat"].std()

In [10]:
def get_num_notes(df: pd.DataFrame):
    return len(df)

In [11]:
def get_pitch_entropy(df: pd.DataFrame):
    notes = df["note"]
    total_notes = len(notes)

    pitches = notes % 12
    counts = pitches.value_counts()

    pitch_entropy = 0.0
    for count in counts:
        p_x = count / total_notes
        pitch_entropy -= p_x * math.log2(p_x)

    return round(pitch_entropy, 4)

In [12]:
def get_transition_matrix(df: pd.DataFrame):
    df_sorted = df.sort_values(by="start_time")

    # Map raw MIDI numbers to the 12 pitch classes (0-11)
    pitches = df_sorted["note"] % 12
    total_notes = len(pitches)

    # Initialize a 12x12 matrix with zeros for counts
    counts = np.zeros((12, 12))

    # Count transitions between consecutive notes
    for i in range(total_notes - 1):
        current_pitch = pitches[i]
        next_pitch = pitches[i + 1]
        counts[current_pitch][next_pitch] += 1

    # Convert absolute counts to row-wise conditional probabilities
    row_sums = counts.sum(axis=1, keepdims=True)
    transition_matrix = np.divide(
        counts, row_sums, out=np.zeros_like(counts), where=row_sums > 0
    )

    note_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    matrix_df = pd.DataFrame(
        transition_matrix, index=note_names, columns=note_names
    )

    return matrix_df.round(4)

In [13]:
def get_rhythmic_entropy(df: pd.DataFrame):
    rhythm_counts = df["note"].value_counts()
    total_rhythms = rhythm_counts.sum()

    rhythmic_entropy = 0.0
    for count in rhythm_counts:
        p_r = count / total_rhythms
        rhythmic_entropy -= p_r * np.log2(p_r)

    return round(rhythmic_entropy, 4)

In [14]:
def get_average_interval_size(df: pd.DataFrame):
    # Ensure chronological order for chronological intervals
    df = df.sort_values(by="start_time")

    midi_notes = df["note"].dropna().values
    avg_interval_size = np.mean(np.abs(np.diff(midi_notes)))
    return round(avg_interval_size, 4)

In [15]:
def get_average_vertical_density(df: pd.DataFrame):
    """
    Measures the average number of physical notes sounding simultaneously
    across all structural time windows.
    """
    # Extract unique timestamps where notes either turn on or off
    times = np.unique(np.concatenate([df["start_time"], df["end_time"]]))
    active_note_counts = []

    # Loop through intervals between consecutive event boundaries
    for i in range(len(times) - 1):
        t_start = times[i]
        t_end = times[i + 1]
        midpoint = t_start + (t_end - t_start) / 2

        # Count active rows spanning across the window's midpoint
        active_rows = df[(df["start_time"] <= midpoint) & (df["end_time"] > midpoint)]
        if not active_rows.empty:
            active_note_counts.append(len(active_rows))

    return round(np.mean(active_note_counts), 4)

In [16]:
def get_average_polyphonic_voices(df: pd.DataFrame):
    """
    Measures the average number of unique, independent horizontal tracks or
    instruments active simultaneously across all structural time windows.
    """
    # Extract unique timestamps where notes either turn on or off
    times = np.unique(np.concatenate([df["start_time"], df["end_time"]]))
    active_instrument_counts = []

    # Loop through intervals between consecutive event boundaries
    for i in range(len(times) - 1):
        t_start = times[i]
        t_end = times[i + 1]
        midpoint = t_start + (t_end - t_start) / 2

        # Find active rows spanning across the window's midpoint
        active_rows = df[(df["start_time"] <= midpoint) & (df["end_time"] > midpoint)]
        if not active_rows.empty:
            if "instrument" in active_rows.columns:
                active_instrument_counts.append(active_rows["instrument"].nunique())
            else:
                active_instrument_counts.append(1)

    return round(np.mean(active_instrument_counts), 4)

In [17]:
def get_fractal_dimension(df, pitch_bins=128, time_bins=100):
    """Estimates the Fractal Dimension of a melody map via the Box-Counting method.

    Projects notes onto a discrete 2D grid matrix of time versus pitch.
    """
    # Normalise dimensions to map into a fixed grid space matrix
    t_min, t_max = df["start_time"].min(), df["end_time"].max()
    p_min, p_max = df["note"].min(), df["note"].max()

    if t_max == t_min or p_max == p_min:
        return 0.0

    # Construct the binary matrix container grid
    grid = np.zeros((pitch_bins, time_bins), dtype=int)

    for _, row in df.iterrows():
        # Scale values proportionally into grid indices
        t1 = int((row["start_time"] - t_min) / (t_max - t_min) * (time_bins - 1))
        t2 = int((row["end_time"] - t_min) / (t_max - t_min) * (time_bins - 1))
        p = int((row["note"] - p_min) / (p_max - p_min) * (pitch_bins - 1))
        grid[p, t1:max(t1 + 1, t2 + 1)] = 1

    # Box-counting calculation sequence across downscaled resolutions
    def count_boxes(size):
        # Divides the grid array into macro blocks and flags active hits
        counts = 0
        for r in range(0, pitch_bins, size):
            for c in range(0, time_bins, size):
                if np.any(grid[r:r + size, c:c + size]):
                    counts += 1
        return counts

    # Calculate scales using factor steps
    sizes = [2, 4, 8, 16]
    counts = [count_boxes(s) for s in sizes]

    # Calculate the linear regression slope in log-log space to find the dimension
    log_sizes = -np.log(sizes)
    log_counts = np.log(counts)

    # Simple slope calculation from points
    slope, _ = np.polyfit(log_sizes, log_counts, 1)
    return round(max(0.0, slope), 4)

In [18]:
def orch_vertical_density_variance(df):
    # Group pitches by start time to evaluate vertical stacking
    pitch_counts = df.groupby('start_time')['note'].nunique()
    if len(pitch_counts) <= 1:
        return 0.0
    return float(np.std(pitch_counts))

In [19]:
def orch_outer_voice_motion_index(df):
    """
    Ignores mid-tier instrument stuffing. Isolates the global orchestral frame 
    by checking counterpoint motion ONLY between the absolute lowest note (Bass) 
    and absolute highest note (Soprano) of the entire orchestra at each time slice.
    """
    slices = df.groupby('start_time')['note'].apply(list).sort_index()
    time_keys = slices.index.tolist()
    
    contrary_oblique_count = 0
    total_transitions = 0
    
    for i in range(len(time_keys) - 1):
        notes_t1 = slices[time_keys[i]]
        notes_t2 = slices[time_keys[i+1]]
        
        if len(notes_t1) >= 2 and len(notes_t2) >= 2:
            # Isolate the extreme margins of the total orchestral sonic field
            p1_low, p1_high = min(notes_t1), max(notes_t1)
            p2_low, p2_high = min(notes_t2), max(notes_t2)
            
            diff_low = p2_low - p1_low
            diff_high = p2_high - p1_high
            
            if (diff_low * diff_high < 0) or (diff_low == 0 and diff_high != 0) or (diff_high == 0 and diff_high != 0):
                contrary_oblique_count += 1
            total_transitions += 1
            
    return (contrary_oblique_count / total_transitions * 100) if total_transitions > 0 else 0.0

In [20]:
def orch_harmonic_change_entropy(df):
    slices = df.groupby('start_time')['note'].apply(lambda x: tuple(sorted(set(x % 12)))).sort_index()
    chord_transitions = [(slices.iloc[i], slices.iloc[i+1]) for i in range(len(slices)-1) if slices.iloc[i] != slices.iloc[i+1]]
    
    if len(chord_transitions) <= 1:
        return 0.0
        
    unique_chords = list(set(slices.values))
    chord_to_idx = {chord: idx for idx, chord in enumerate(unique_chords)}
    
    matrix = np.zeros((len(unique_chords), len(unique_chords)))
    for c1, c2 in chord_transitions:
        matrix[chord_to_idx[c1], chord_to_idx[c2]] += 1
        
    row_sums = matrix.sum(axis=1, keepdims=True)
    matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums!=0)
    
    all_valid_chords = [c for c1, c2 in chord_transitions for c in [c1]]
    state_counts = np.array([all_valid_chords.count(c) for c in unique_chords])
    
    if state_counts.sum() == 0:
        return 0.0
    pi = state_counts / state_counts.sum()
    
    entropy = 0.0
    for i in range(len(unique_chords)):
        row = matrix[i, :]
        row_entropy = -np.sum(row * np.log2(row, out=np.zeros_like(row), where=row>0))
        entropy += pi[i] * row_entropy
        
    return entropy

In [21]:
def orch_global_npvi(df):
    unique_onsets = sorted(df['start_time'].unique())
    if len(unique_onsets) <= 2:
        return 0.0
        
    # Find the duration length of every macro-orchestral rhythmic block
    global_durations = np.diff(unique_onsets)
    m = len(global_durations)
    
    npvi_sum = 0.0
    for k in range(m - 1):
        d_k = global_durations[k]
        d_next = global_durations[k+1]
        if (d_k + d_next) > 0:
            npvi_sum += abs(d_k - d_next) / ((d_k + d_next) / 2.0)
            
    return (100.0 / (m - 1)) * npvi_sum

In [22]:
# print(f"""Pitch entropy: {get_pitch_entropy(df)}
# Rhythmic entropy: {get_rhythmic_entropy(df)}
# Average interval size: {get_average_interval_size(df)}
# Average vertical density: {get_average_vertical_density(df)}
# Average polyphonic voices: {get_average_polyphonic_voices(df)}
# Fractal dimension: {get_fractal_dimension(df)}
# """)

# df = pd.read_csv("data/musicnet/musicnet/train_labels/2677.csv")
# print(orch_global_npvi(df))

In [206]:
composer_folder_path = Path("data/musicnet_midis/musicnet_midis")

file_to_composer = {}

for folder_path in composer_folder_path.iterdir():
    for file_path in folder_path.iterdir():
        if file_path.is_file() and "test" not in str(file_path):
            file_to_composer[file_path.name[:4]] = str(file_path)[35:-str(file_path)[::-1].index("/")-1]
            
print(file_to_composer)

{'2076': 'Cambini', '2075': 'Cambini', '2082': 'Cambini', '2077': 'Cambini', '2079': 'Cambini', '2083': 'Cambini', '2080': 'Cambini', '2081': 'Cambini', '2078': 'Cambini', '2528': 'Beethoven', '2492': 'Beethoven', '2538': 'Beethoven', '2595': 'Beethoven', '2677': 'Beethoven', '2391': 'Beethoven', '2342': 'Beethoven', '2341': 'Beethoven', '2488': 'Beethoven', '2486': 'Beethoven', '2566': 'Beethoven', '2314': 'Beethoven', '2537': 'Beethoven', '2627': 'Beethoven', '2472': 'Beethoven', '2444': 'Beethoven', '2423': 'Beethoven', '2507': 'Beethoven', '2505': 'Beethoven', '2410': 'Beethoven', '2506': 'Beethoven', '2442': 'Beethoven', '2462': 'Beethoven', '2374': 'Beethoven', '2620': 'Beethoven', '2415': 'Beethoven', '2364': 'Beethoven', '2390': 'Beethoven', '2320': 'Beethoven', '2568': 'Beethoven', '2473': 'Beethoven', '2608': 'Beethoven', '2373': 'Beethoven', '2622': 'Beethoven', '2318': 'Beethoven', '2335': 'Beethoven', '2530': 'Beethoven', '2596': 'Beethoven', '2336': 'Beethoven', '2607': '

In [207]:
# Specify your folder path
folder_path = Path("data/musicnet/musicnet/train_labels")

In [208]:
with open("train_data.csv", mode='w') as file:
    file.write("file_name,composer,composer_label,pitch_entropy,rhythmic_entropy,average_interval_size,average_vertical_density,average_polyphonic_voices,fractal_dimension,num_notes,pitch_std,pitch_range,number_of_instruments,unique_note_values,average_beat_gap,most_common_note,average_note_length,largest_leap,orch_outer_voice_motion_index,orch_harmonic_change_entropy,orch_global_npvi\n")
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            df = pd.read_csv(file_path)
            file.write(f"{file_path.name},{file_to_composer[file_path.name.strip('.csv')]},{composer_to_label[file_to_composer[file_path.name.strip('.csv')]]},{get_pitch_entropy(df)},{get_rhythmic_entropy(df)},{get_average_interval_size(df)},{get_average_vertical_density(df)},{get_average_polyphonic_voices(df)},{get_fractal_dimension(df)},{get_num_notes(df)},{get_pitch_std(df)},{get_pitch_range(df)},{get_number_of_instruments(df)},{get_unique_note_values(df)},{get_average_beat_gap(df)},{get_most_common_note(df)},{get_average_note_length(df)},{get_largest_leap(df)},{orch_outer_voice_motion_index(df)},{orch_harmonic_change_entropy(df)},{orch_global_npvi(df)}\n")
            print("Wrote data for", file_path)
print("Done!")

Wrote data for data/musicnet/musicnet/train_labels/2677.csv
Wrote data for data/musicnet/musicnet/train_labels/2147.csv
Wrote data for data/musicnet/musicnet/train_labels/2607.csv
Wrote data for data/musicnet/musicnet/train_labels/1751.csv
Wrote data for data/musicnet/musicnet/train_labels/2471.csv
Wrote data for data/musicnet/musicnet/train_labels/2504.csv
Wrote data for data/musicnet/musicnet/train_labels/1775.csv
Wrote data for data/musicnet/musicnet/train_labels/2307.csv
Wrote data for data/musicnet/musicnet/train_labels/2156.csv
Wrote data for data/musicnet/musicnet/train_labels/2568.csv
Wrote data for data/musicnet/musicnet/train_labels/2204.csv
Wrote data for data/musicnet/musicnet/train_labels/1760.csv
Wrote data for data/musicnet/musicnet/train_labels/2480.csv
Wrote data for data/musicnet/musicnet/train_labels/2410.csv
Wrote data for data/musicnet/musicnet/train_labels/1817.csv
Wrote data for data/musicnet/musicnet/train_labels/2537.csv
Wrote data for data/musicnet/musicnet/tr

In [202]:
composer_to_label = {
    "Bach": 0, 
    "Beethoven": 1, 
    "Brahms": 2, 
    "Cambini": 3, 
    "Dvorak": 4, 
    "Faure": 5, 
    "Haydn": 6, 
    "Mozart": 7, 
    "Ravel": 8, 
    "Schubert": 9
}

In [203]:
composer_folder_path = Path("data/musicnet_midis/musicnet_midis")

file_to_composer = {}

for folder_path in composer_folder_path.iterdir():
    for file_path in folder_path.iterdir():
        if file_path.is_file() and "test" in str(file_path):
            file_to_composer[file_path.name[:4]] = str(file_path)[35:-str(file_path)[::-1].index("/")-1]
            
print(file_to_composer)

{'2298': 'test', '2106': 'test', '2382': 'test', '2191': 'test', '2628': 'test', '1759': 'test', '2416': 'test', '2556': 'test', '2303': 'test', '1819': 'test'}


In [204]:
# Specify your folder path
test_folder_path = Path("data/musicnet/musicnet/test_labels")
#df2 = pd.read_csv("data/musicnet_metadata.csv")
#print(df2[df2['id'] == 1759]['composer'].iloc[0])


In [205]:
with open("test_data.csv", mode='w') as file:
    file.write("file_name,pitch_entropy,rhythmic_entropy,average_interval_size,average_vertical_density,average_polyphonic_voices,fractal_dimension,num_notes,pitch_std,pitch_range,number_of_instruments,unique_note_values,average_beat_gap,most_common_note,average_note_length,largest_leap,orch_outer_voice_motion_index,orch_harmonic_change_entropy,orch_global_npvi\n")
    for file_path in test_folder_path.iterdir():
        if file_path.is_file():
            df = pd.read_csv(file_path)
            df2 = pd.read_csv("data/musicnet_metadata.csv")
            file.write(f"{file_path.name},{get_pitch_entropy(df)},{get_rhythmic_entropy(df)},{get_average_interval_size(df)},{get_average_vertical_density(df)},{get_average_polyphonic_voices(df)},{get_fractal_dimension(df)},{get_num_notes(df)},{get_pitch_std(df)},{get_pitch_range(df)},{get_number_of_instruments(df)},{get_unique_note_values(df)},{get_average_beat_gap(df)},{get_most_common_note(df)},{get_average_note_length(df)},{get_largest_leap(df)},{orch_outer_voice_motion_index(df)},{orch_harmonic_change_entropy(df)},{orch_global_npvi(df)}\n")
            print("Wrote data for", file_path)
print("Done!")

Wrote data for data/musicnet/musicnet/test_labels/2303.csv
Wrote data for data/musicnet/musicnet/test_labels/2106.csv
Wrote data for data/musicnet/musicnet/test_labels/2382.csv
Wrote data for data/musicnet/musicnet/test_labels/2556.csv
Wrote data for data/musicnet/musicnet/test_labels/2416.csv
Wrote data for data/musicnet/musicnet/test_labels/2298.csv
Wrote data for data/musicnet/musicnet/test_labels/1819.csv
Wrote data for data/musicnet/musicnet/test_labels/2191.csv
Wrote data for data/musicnet/musicnet/test_labels/2628.csv
Wrote data for data/musicnet/musicnet/test_labels/1759.csv
Done!
